# EEG Motor Imagery BCI 분석

PhysioNet EEG Motor Movement/Imagery(EEGBCI) 데이터셋으로 **왼손/오른손 운동상상(Motor Imagery)**을 분류하는 포트폴리오용 분석 노트북입니다.

핵심 흐름:

`EEGBCI → 8–30 Hz 필터링 → Epoch → CSP → LDA → 개인별 평가 → Nested CV → LOSO → Calibration → CSP Topomap`

- **Run All** 시 오래 걸리는 대규모 실험은 자동 실행하지 않습니다.
- 기본 셀은 환경/함수/기존 결과 기록과 결과 그래프만 준비합니다.
- 오래 걸리는 재평가는 아래 **선택 실행** 섹션의 스위치를 `True`로 바꿔 실행합니다.
- EEGBCI 데이터 경로: `D:\bci_data`


## 1. 환경 설정

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from mne.datasets import eegbci
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_val_score

# 그래프 한글 표시
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

DATA_PATH = Path(r"D:\bci_data")
RUNS = [4, 8, 12]          # 왼손 / 오른손 Motor Imagery
FILTER_BAND = (8, 30)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"EEG 데이터 폴더가 없습니다: {DATA_PATH}")

print("환경 설정 완료")
print("MNE 버전:", mne.__version__)
print("데이터 경로:", DATA_PATH)


## 2. 공통 설정

In [ ]:
# 고정 설정 평가에 사용한 기준
DEFAULT_TMIN = 0.0
DEFAULT_TMAX = 3.0
DEFAULT_CSP_COMPONENTS = 4

# 파라미터 탐색 범위
TIME_WINDOWS = [
    (0.0, 4.0),
    (0.0, 3.0),
    (0.5, 3.5),
    (1.0, 4.0),
    (1.0, 3.0),
    (1.5, 4.0),
    (2.0, 4.0),
]
CSP_COMPONENTS = [2, 4, 6, 8, 10, 12]

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
INNER_CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
OUTER_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("공통 실험 설정 완료")

## 3. 데이터 로딩 / 전처리 함수

In [ ]:
def load_subject_epochs(subject_number, tmin=0.0, tmax=4.0):
    """
    EEGBCI의 Subject 1명에 대해 R04/R08/R12를 불러와
    8~30 Hz 필터링 후 왼손/오른손 Epoch를 합쳐 반환합니다.
    """
    files = eegbci.load_data(
        subject_number,
        RUNS,
        path=str(DATA_PATH),
        verbose=False,
    )

    subject_epochs = []

    for file in files:
        raw = mne.io.read_raw_edf(file, preload=True, verbose=False)
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        event_id_lr = {
            "left": event_id["T1"],
            "right": event_id["T2"],
        }

        epochs = mne.Epochs(
            raw,
            events,
            event_id=event_id_lr,
            tmin=tmin,
            tmax=tmax,
            baseline=None,
            preload=True,
            verbose=False,
        )

        epochs.filter(
            l_freq=FILTER_BAND[0],
            h_freq=FILTER_BAND[1],
            verbose=False,
        )

        subject_epochs.append(epochs)

    return mne.concatenate_epochs(subject_epochs, verbose=False)


def make_csp_lda_model(n_components=4):
    """CSP + LDA 분류 파이프라인을 생성합니다."""
    return Pipeline([
        (
            "CSP",
            CSP(
                n_components=n_components,
                reg=None,
                log=True,
                norm_trace=False,
            ),
        ),
        ("LDA", LinearDiscriminantAnalysis()),
    ])

print("데이터/모델 함수 정의 완료")

## 4. 고정 설정 평가 함수

In [ ]:
def evaluate_subject(subject_number, verbose=True):
    """0~3초, CSP 4개 고정 설정으로 5-Fold CV 성능을 평가합니다."""
    epochs = load_subject_epochs(subject_number, tmin=0.0, tmax=4.0)
    epochs.crop(tmin=DEFAULT_TMIN, tmax=DEFAULT_TMAX)

    X = epochs.get_data()
    y = epochs.events[:, -1]

    model = make_csp_lda_model(DEFAULT_CSP_COMPONENTS)
    scores = cross_val_score(model, X, y, cv=CV, scoring="accuracy")

    if verbose:
        print(f"=== Subject {subject_number} 고정 설정 ===")
        for i, score in enumerate(scores, 1):
            print(f"{i}번째 Fold: {score * 100:.1f}%")
        print(f"평균 정확도: {scores.mean() * 100:.1f}%")
        print(f"표준편차: {scores.std() * 100:.1f}%")

    return scores

## 5. 피험자별 파라미터 탐색 함수

> 이 함수의 최고 정확도는 **탐색 최고점**입니다. 같은 데이터로 여러 설정을 비교해 최고값을 선택하므로 최종 일반화 성능으로 사용하지 않습니다.

In [ ]:
def optimize_subject(subject_number, verbose=True):
    """시간 구간 × CSP 컴포넌트 조합을 탐색합니다."""
    epochs_subject = load_subject_epochs(subject_number, tmin=0.0, tmax=4.0)
    y = epochs_subject.events[:, -1]
    results = []

    for tmin, tmax in TIME_WINDOWS:
        epochs_window = epochs_subject.copy().crop(tmin=tmin, tmax=tmax)
        X = epochs_window.get_data()

        for n_components in CSP_COMPONENTS:
            model = make_csp_lda_model(n_components)
            scores = cross_val_score(model, X, y, cv=CV, scoring="accuracy")

            results.append({
                "시간": f"{tmin:.1f}~{tmax:.1f}",
                "tmin": tmin,
                "tmax": tmax,
                "CSP": n_components,
                "평균": scores.mean(),
                "표준편차": scores.std(),
            })

    best = max(results, key=lambda x: x["평균"])

    if verbose:
        print(f"=== Subject {subject_number} 최적화 결과 ===")
        print(f"시간 구간: {best['시간']}초")
        print(f"CSP 컴포넌트: {best['CSP']}개")
        print(f"평균 정확도: {best['평균'] * 100:.1f}%")
        print(f"표준편차: {best['표준편차'] * 100:.1f}%")

    return results, best

## 6. Nested Cross-Validation 함수

설정 선택용 Inner CV와 최종 평가용 Outer CV를 분리해 탐색에 의한 과대평가를 줄입니다.

In [ ]:
def nested_evaluate_subject(subject_number, verbose=True):
    """개인별 시간 구간/CSP 탐색을 Nested CV로 평가합니다."""
    epochs_subject = load_subject_epochs(subject_number, tmin=0.0, tmax=4.0)
    y = epochs_subject.events[:, -1]

    outer_scores = []
    chosen_params = []

    for outer_fold, (train_idx, test_idx) in enumerate(
        OUTER_CV.split(np.zeros(len(y)), y),
        start=1,
    ):
        best_inner_score = -1
        best_params = None

        for tmin, tmax in TIME_WINDOWS:
            epochs_window = epochs_subject.copy().crop(tmin=tmin, tmax=tmax)
            X_window = epochs_window.get_data()
            X_train_outer = X_window[train_idx]
            y_train_outer = y[train_idx]

            for n_components in CSP_COMPONENTS:
                model_inner = make_csp_lda_model(n_components)
                inner_scores = cross_val_score(
                    model_inner,
                    X_train_outer,
                    y_train_outer,
                    cv=INNER_CV,
                    scoring="accuracy",
                )
                mean_inner = inner_scores.mean()

                if mean_inner > best_inner_score:
                    best_inner_score = mean_inner
                    best_params = {
                        "tmin": tmin,
                        "tmax": tmax,
                        "csp": n_components,
                    }

        epochs_best = epochs_subject.copy().crop(
            tmin=best_params["tmin"],
            tmax=best_params["tmax"],
        )
        X_best = epochs_best.get_data()

        X_train, X_test = X_best[train_idx], X_best[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        final_model = make_csp_lda_model(best_params["csp"])
        final_model.fit(X_train, y_train)
        outer_accuracy = final_model.score(X_test, y_test)

        outer_scores.append(outer_accuracy)
        chosen_params.append(best_params)

        if verbose:
            print(
                f"Subject {subject_number} | Outer Fold {outer_fold}: "
                f"{outer_accuracy * 100:.1f}% | "
                f"{best_params['tmin']:.1f}~{best_params['tmax']:.1f}초 | "
                f"CSP {best_params['csp']}개"
            )

    result = {
        "subject": subject_number,
        "mean": float(np.mean(outer_scores)),
        "std": float(np.std(outer_scores)),
        "fold_scores": outer_scores,
        "params": chosen_params,
    }

    if verbose:
        print()
        print(f"=== Subject {subject_number} Nested CV ===")
        print(f"평균 정확도: {result['mean'] * 100:.1f}%")
        print(f"표준편차: {result['std'] * 100:.1f}%")

    return result

## 7. Subject-independent 데이터 준비 / LOSO

**LOSO(Leave-One-Subject-Out)**는 테스트할 피험자의 데이터를 학습에 전혀 사용하지 않고,
나머지 피험자들로 학습한 모델이 새로운 사용자에게 일반화되는지 평가합니다.


In [ ]:
def load_subject_data(subject_number, tmin=DEFAULT_TMIN, tmax=DEFAULT_TMAX):
    """피험자 1명의 전처리 EEG를 X, y 배열로 반환합니다."""
    epochs = load_subject_epochs(subject_number, tmin=0.0, tmax=4.0)
    epochs.crop(tmin=tmin, tmax=tmax)
    X = epochs.get_data()
    y = epochs.events[:, -1]
    return X, y


def build_subject_data(subjects=range(1, 11)):
    """여러 피험자의 X, y를 메모리에 준비합니다."""
    data = {}
    for subject in subjects:
        X, y = load_subject_data(subject)
        data[subject] = {"X": X, "y": y}
        print(f"Subject {subject}: X {X.shape}, y {y.shape}")
    return data


def loso_evaluate(subject_data, n_components=DEFAULT_CSP_COMPONENTS):
    """한 명씩 완전히 제외하고 나머지 피험자 데이터로 학습하여 평가합니다."""
    results = {}
    subjects = sorted(subject_data)

    for test_subject in subjects:
        X_test = subject_data[test_subject]["X"]
        y_test = subject_data[test_subject]["y"]

        X_train = np.concatenate([
            subject_data[s]["X"] for s in subjects if s != test_subject
        ], axis=0)
        y_train = np.concatenate([
            subject_data[s]["y"] for s in subjects if s != test_subject
        ], axis=0)

        model = make_csp_lda_model(n_components)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        results[test_subject] = float(score)

        print(f"Subject {test_subject}: {score * 100:.1f}%")

    print(f"\nLOSO 평균 정확도: {np.mean(list(results.values())) * 100:.1f}%")
    return results


## 8. Calibration 실험

새로운 사용자의 데이터를 `0 / 5 / 10 / 20 trial`만큼 학습 데이터에 추가했을 때
성능이 얼마나 회복되는지 확인합니다.

> Calibration 데이터 수가 증가하면 남은 테스트 trial 수는 감소하므로, 이 결과는 해당 조건을 함께 고려해 해석합니다.


In [ ]:
def calibration_experiment(
    test_subject,
    subject_data,
    calibration_sizes=(0, 5, 10, 20),
    repeats=10,
    random_state=42,
):
    """타인 데이터 + 본인 일부 calibration trial로 학습 후 남은 본인 trial을 평가합니다."""
    X_subject = subject_data[test_subject]["X"]
    y_subject = subject_data[test_subject]["y"]
    subjects = sorted(subject_data)

    X_other = np.concatenate([
        subject_data[s]["X"] for s in subjects if s != test_subject
    ], axis=0)
    y_other = np.concatenate([
        subject_data[s]["y"] for s in subjects if s != test_subject
    ], axis=0)

    results = {}

    for calibration_size in calibration_sizes:
        scores = []

        if calibration_size == 0:
            model = make_csp_lda_model(DEFAULT_CSP_COMPONENTS)
            model.fit(X_other, y_other)
            scores.append(model.score(X_subject, y_subject))
        else:
            splitter = StratifiedShuffleSplit(
                n_splits=repeats,
                train_size=calibration_size,
                random_state=random_state,
            )

            for calibration_idx, test_idx in splitter.split(X_subject, y_subject):
                X_train = np.concatenate(
                    [X_other, X_subject[calibration_idx]], axis=0
                )
                y_train = np.concatenate(
                    [y_other, y_subject[calibration_idx]], axis=0
                )

                model = make_csp_lda_model(DEFAULT_CSP_COMPONENTS)
                model.fit(X_train, y_train)
                scores.append(model.score(X_subject[test_idx], y_subject[test_idx]))

        results[calibration_size] = {
            "mean": float(np.mean(scores)),
            "std": float(np.std(scores)),
        }

    return results


## 9. CSP 공간 패턴(Topomap)

CSP가 분류에 사용한 공간 패턴을 머리 위 전극 분포로 시각화합니다.
EEGBCI 채널명은 `eegbci.standardize()`로 표준화한 뒤 `standard_1005` montage를 적용합니다.

색은 **왼손/오른손 라벨 자체가 아니라 CSP 패턴의 방향과 크기**를 의미합니다.


In [ ]:
def load_subject_epochs_with_montage(subject_number, tmin=0.0, tmax=3.0):
    """Topomap용 채널 위치 정보를 포함한 Epoch를 반환합니다."""
    files = eegbci.load_data(
        subject_number,
        RUNS,
        path=str(DATA_PATH),
        verbose=False,
    )

    montage = mne.channels.make_standard_montage("standard_1005")
    epochs_list = []

    for file in files:
        raw = mne.io.read_raw_edf(file, preload=True, verbose=False)

        # EEGBCI 전용 표준 채널명 변환
        eegbci.standardize(raw)
        raw.set_montage(montage, on_missing="raise")

        events, event_id = mne.events_from_annotations(raw, verbose=False)
        event_id_lr = {
            "left": event_id["T1"],
            "right": event_id["T2"],
        }

        epochs = mne.Epochs(
            raw,
            events,
            event_id=event_id_lr,
            tmin=tmin,
            tmax=tmax,
            baseline=None,
            preload=True,
            verbose=False,
        )
        epochs.filter(*FILTER_BAND, verbose=False)
        epochs_list.append(epochs)

    return mne.concatenate_epochs(epochs_list, verbose=False)


def plot_csp_patterns(subject_number=7, n_components=4):
    """피험자의 CSP 상위 공간 패턴을 Topomap으로 표시합니다."""
    from mne.decoding import get_spatial_filter_from_estimator

    epochs = load_subject_epochs_with_montage(subject_number)
    X = epochs.get_data()
    y = epochs.events[:, -1]

    csp = CSP(
        n_components=n_components,
        reg=None,
        log=True,
        norm_trace=False,
    )
    csp.fit(X, y)

    spatial_filter = get_spatial_filter_from_estimator(csp, info=epochs.info)
    return spatial_filter.plot_patterns(
        components=list(range(n_components)),
        ch_type="eeg",
        size=1.5,
    )


## 10. 지금까지 얻은 결과 기록

아래 값은 이전 노트북에서 이미 계산한 결과입니다. **Run All 시 재계산하지 않고 기록만 불러옵니다.**


In [ ]:
fixed_accuracy = {
    1: 71.1, 2: 55.6, 3: 57.8, 4: 51.1, 5: 48.9,
    6: 46.7, 7: 95.6, 8: 48.9, 9: 40.0, 10: 53.3,
}

optimized_accuracy = {
    2: 91.1, 3: 75.6, 4: 66.7, 5: 62.2, 6: 62.2,
    7: 100.0, 8: 66.7, 9: 66.7, 10: 80.0,
}

nested_summary = {
    1: (51.1, 11.3),
    2: (82.2, 8.9),
    3: (68.9, 13.0),
    4: (51.1, 18.1),
    5: (44.4, 15.7),
    6: (48.9, 8.9),
    7: (93.3, 8.9),
    8: (53.3, 13.0),
    9: (48.9, 20.6),
    10: (55.6, 23.3),
}

loso_accuracy = {
    1: 64.4, 2: 53.3, 3: 44.4, 4: 60.0, 5: 48.9,
    6: 48.9, 7: 55.6, 8: 57.8, 9: 46.7, 10: 64.4,
}

calibration_mean = {
    0: 54.4,
    5: 54.6,
    10: 57.8,
    20: 59.2,
}

calibration_subject_std = {
    0: 6.8,
    5: 4.7,
    10: 7.3,
    20: 8.5,
}

subject7_calibration = {
    0: (55.6, 0.0),
    5: (61.5, 5.9),
    10: (69.4, 7.7),
    20: (76.8, 11.8),
}

rows = []
for subject in range(1, 11):
    nested_mean, nested_std = nested_summary[subject]
    rows.append({
        "Subject": subject,
        "고정 설정 (%)": fixed_accuracy.get(subject, np.nan),
        "탐색 최고점 (%)": optimized_accuracy.get(subject, np.nan),
        "Nested CV (%)": nested_mean,
        "Nested 표준편차 (%)": nested_std,
        "LOSO (%)": loso_accuracy[subject],
    })

results_df = pd.DataFrame(rows)
results_df


## 11. 포트폴리오용 결과 시각화

아래 그래프는 저장해 둔 실험 결과를 사용하므로 **Run All 시 재학습 없이 바로 생성**됩니다.


In [ ]:
# 1) 개인별 Nested CV vs LOSO
subjects = list(range(1, 11))
nested_scores = [nested_summary[s][0] for s in subjects]
loso_scores = [loso_accuracy[s] for s in subjects]

x = np.arange(len(subjects))
width = 0.35

plt.figure(figsize=(12, 6))
plt.bar(x - width/2, nested_scores, width, label="개인별 Nested CV")
plt.bar(x + width/2, loso_scores, width, label="LOSO")
plt.axhline(50, linestyle="--", color="gray", label="Chance level (50%)")
plt.xlabel("Subject")
plt.ylabel("정확도 (%)")
plt.title("개인별 모델과 Subject-independent 모델 성능 비교")
plt.xticks(x, subjects)
plt.ylim(30, 100)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 2) 전체 평균 Calibration curve
sizes = list(calibration_mean.keys())
means = [calibration_mean[s] for s in sizes]
stds = [calibration_subject_std[s] for s in sizes]

plt.figure(figsize=(9, 5))
plt.plot(sizes, means, marker="o", linewidth=3, label="전체 평균")
plt.fill_between(
    sizes,
    np.array(means) - np.array(stds),
    np.array(means) + np.array(stds),
    alpha=0.15,
    label="Subject 간 표준편차",
)
plt.axhline(50, linestyle="--", color="gray", label="Chance level (50%)")
plt.xlabel("Calibration 데이터 수")
plt.ylabel("분류 정확도 (%)")
plt.title("개인 Calibration 데이터 수에 따른 평균 분류 정확도")
plt.xticks(sizes)
plt.ylim(40, 75)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


### 현재 핵심 결과 요약

- 개인별 Nested CV 평균: 약 **59.8%**
- LOSO 평균: **54.4%**
- Calibration 평균: **54.4% → 54.6% → 57.8% → 59.2%** (`0 → 5 → 10 → 20 trials`)
- Subject 7 Calibration: **55.6% → 76.8%** (`0 → 20 trials`)
- 해석: 새로운 사용자에 대한 일반화는 제한적이었고, 개인 calibration은 평균적으로 성능을 개선했지만 효과 크기에는 피험자별 차이가 컸습니다.


## 12. 선택 실행 — 오래 걸리는 실험

아래 셀은 **Run All 해도 자동으로 계산하지 않도록 기본값이 `False`**입니다. 필요할 때 `True`로 바꿔 실행하세요.


In [ ]:
RUN_LOSO_1_TO_10 = False

if RUN_LOSO_1_TO_10:
    subject_data = build_subject_data(range(1, 11))
    loso_results_new = loso_evaluate(subject_data)


In [ ]:
RUN_CALIBRATION_1_TO_10 = False

if RUN_CALIBRATION_1_TO_10:
    if "subject_data" not in globals():
        subject_data = build_subject_data(range(1, 11))

    all_calibration_results = {}
    for subject in range(1, 11):
        all_calibration_results[subject] = calibration_experiment(
            subject,
            subject_data,
            calibration_sizes=(0, 5, 10, 20),
            repeats=10,
        )
        print(f"Subject {subject} 완료")


In [ ]:
RUN_CSP_TOPOMAP = False

if RUN_CSP_TOPOMAP:
    plot_csp_patterns(subject_number=7, n_components=4)


In [ ]:
RUN_FIXED_1_TO_10 = False

if RUN_FIXED_1_TO_10:
    fixed_results_new = {}
    for subject in range(1, 11):
        scores = evaluate_subject(subject, verbose=True)
        fixed_results_new[subject] = scores
else:
    print("고정 설정 1~10 재평가: 건너뜀")

In [ ]:
RUN_OPTIMIZE_1_TO_10 = False

if RUN_OPTIMIZE_1_TO_10:
    optimized_results_new = {}
    for subject in range(1, 11):
        print("\n" + "=" * 60)
        _, best = optimize_subject(subject, verbose=True)
        optimized_results_new[subject] = best
else:
    print("피험자별 파라미터 탐색 1~10: 건너뜀")

In [ ]:
RUN_NESTED_1_TO_10 = False

if RUN_NESTED_1_TO_10:
    nested_results_new = {}
    for subject in range(1, 11):
        print("\n" + "=" * 60)
        nested_results_new[subject] = nested_evaluate_subject(subject, verbose=True)
else:
    print("Nested CV 1~10 재평가: 건너뜀")